# Phase 7 — C2: the unavoidable-error ceiling

**Why (paper §3.10):** our labels are **daily averages**, but each photo is an **instant**.
Pollution genuinely swings within a day, so even a *perfect* model reading the exact
instantaneous pollution from a photo would be "wrong" versus a daily-average label. That
gap is noise no model can beat — a hard ceiling on achievable R².

We estimate it as **R²_max = 1 − Var(ε) / Var(y)**, where `Var(y)` is the variance of our
daily-average AQI labels and `Var(ε)` is the *average within-day variance* of AQI, measured
from **hourly** reference data (OpenAQ). This tells us how much room actually remains above
the published R²=0.55.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh Colab session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

## Get a free OpenAQ API key (2 minutes)

1. Go to **https://explore.openaq.org** → sign up (free).
2. Open your **account → API keys** and copy your key.
3. Paste it below. (It's kept only in this session — don't commit it.)

In [ ]:
OPENAQ_API_KEY = "PASTE_YOUR_OPENAQ_KEY_HERE"   # <-- from explore.openaq.org
assert OPENAQ_API_KEY != "PASTE_YOUR_OPENAQ_KEY_HERE", "Add your OpenAQ API key first." 

## Fetch hourly PM2.5 from a global sample of stations

We pull ~40 stations' hourly readings over a recent 45-day window. Exact matching to
PM25Vision's stations isn't needed — we want a defensible estimate of typical within-day
AQI variance.

In [ ]:
import datetime as dt
from src import ceiling as CE
to = dt.date.today()
frm = to - dt.timedelta(days=45)
hourly = CE.fetch_openaq_hourly(OPENAQ_API_KEY,
                                date_from=frm.isoformat(), date_to=to.isoformat(),
                                n_locations=40)
print("hourly rows fetched:", len(hourly), "| stations:", hourly["location_id"].nunique())
hourly.head()

## Compute Var(ε) and the ceiling

In [ ]:
from src.config import load_config
from src import data
cfg = load_config()

var_eps, per_day = CE.within_day_aqi_variance(hourly)
_, df = data.load_clean(cfg["data"]["drive_path"], from_disk=True, seed=cfg["seed"])
var_labels = float(df["pm25"].var())

res = CE.error_ceiling(var_eps, var_labels)
print("Var(eps)  (within-day AQI variance): %.1f  (std ~ %.1f AQI)" % (var_eps, var_eps**0.5))
print("Var(y)    (dataset label variance):  %.1f  (std ~ %.1f AQI)" % (var_labels, var_labels**0.5))
print("--------")
print("R2_max (best achievable R2): %.3f" % res["R2_max"])
print("published baseline R2:       0.550")
print("honest interval-width floor: ~%.1f AQI (a range narrower than this over-claims)" % res["interval_width_floor"])

## Save for the paper

In [ ]:
import os, json
os.makedirs(cfg["data"]["outputs_dir"], exist_ok=True)
json.dump({**res, "n_station_days": int(len(per_day))},
          open(os.path.join(cfg["data"]["outputs_dir"], "error_ceiling.json"), "w"), indent=2)
print("saved error_ceiling.json")

## Reading the result

- If **R²_max is well above 0.55**, there's real room to improve over the published baseline.
- If it's **close to 0.55**, the published result may already be near the best possible given
  daily-average labels — itself a strong, honest finding for the paper.
- The **width floor** sets a lower bound on honest interval width: no interval should be
  narrower than the label noise itself.

Paste these numbers back and I'll write them into `docs/RESULTS.md`.

**Next:** `08_abstention.ipynb` (C3 — refusing to answer on unusable inputs).